# Capítulo 7: Construindo Aplicações de Chat
## Introdução Rápida à API OpenAI

Este notebook é adaptado do [Repositório de Exemplos Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) que inclui notebooks que acessam serviços [Azure OpenAI](notebook-azure-openai.ipynb).

A API OpenAI em Python também funciona com Modelos Azure OpenAI, com algumas modificações. Saiba mais sobre as diferenças aqui: [Como alternar entre endpoints OpenAI e Azure OpenAI com Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Visão geral  
"Grandes modelos de linguagem são funções que mapeiam texto para texto. Dada uma string de texto de entrada, um grande modelo de linguagem tenta prever o texto que virá a seguir"(1). Este notebook de "início rápido" apresentará aos usuários conceitos de LLM de alto nível, requisitos principais do pacote para começar com AML, uma introdução suave ao design de prompts e vários exemplos curtos de diferentes casos de uso. 


## Índice  

[Visão Geral](#overview)  
[Como usar o Serviço OpenAI](#how-to-use-openai-service)  
[1. Criando seu Serviço OpenAI](#1.-creating-your-openai-service)  
[2. Instalação](#2.-installation)    
[3. Credenciais](#3.-credentials)  

[Casos de Uso](#use-cases)    
[1. Resumir Texto](#1.-summarize-text)  
[2. Classificar Texto](#2.-classify-text)  
[3. Gerar Novos Nomes de Produtos](#3.-generate-new-product-names)  
[4. Ajustar Fino um Classificador](#4.fine-tune-a-classifier)  

[Referências](#references)


### Construa seu primeiro prompt  
Este exercício curto fornecerá uma introdução básica para enviar prompts a um modelo da OpenAI para uma tarefa simples de "resumo".


**Passos**:  
1. Instale a biblioteca OpenAI no seu ambiente Python  
2. Carregue bibliotecas auxiliares padrão e configure suas credenciais de segurança típicas da OpenAI para o Serviço OpenAI que você criou  
3. Escolha um modelo para sua tarefa  
4. Crie um prompt simples para o modelo  
5. Envie sua solicitação para a API do modelo!


### 1. Instale o OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Importar bibliotecas auxiliares e instanciar credenciais


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Encontrando o modelo certo  
Modelos como GPT-4o e GPT-4o mini podem entender e gerar linguagem natural, e estão disponíveis na plataforma OpenAI com diferentes níveis de poder e velocidade adequados para diferentes tarefas.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Design de Prompt  

"A magia dos grandes modelos de linguagem é que, ao serem treinados para minimizar esse erro de previsão em vastas quantidades de texto, os modelos acabam aprendendo conceitos úteis para essas previsões. Por exemplo, eles aprendem conceitos como"(1):

* como soletrar
* como a gramática funciona
* como parafrasear
* como responder perguntas
* como manter uma conversa
* como escrever em várias línguas
* como programar
* etc.

#### Como controlar um grande modelo de linguagem  
"De todas as entradas para um grande modelo de linguagem, de longe a mais influente é o prompt de texto(1).

Grandes modelos de linguagem podem ser solicitados a produzir uma saída de algumas maneiras:

Instrução: Diga ao modelo o que você quer
Completação: Induza o modelo a completar o início do que você quer
Demonstração: Mostre ao modelo o que você quer, com:
Alguns exemplos no prompt
Centenas ou milhares de exemplos em um conjunto de dados para treinamento fino"



#### Há três diretrizes básicas para criar prompts:

**Mostre e conte**. Deixe claro o que você quer, seja por meio de instruções, exemplos ou uma combinação dos dois. Se você quer que o modelo classifique uma lista de itens em ordem alfabética ou classifique um parágrafo por sentimento, mostre que é isso que você quer.

**Forneça dados de qualidade**. Se você está tentando construir um classificador ou fazer o modelo seguir um padrão, certifique-se de que há exemplos suficientes. Tenha certeza de revisar seus exemplos — o modelo geralmente é inteligente o bastante para perceber erros básicos de ortografia e fornecer uma resposta, mas também pode assumir que isso é intencional e isso pode afetar a resposta.

**Verifique suas configurações.** As configurações temperature e top_p controlam o quão determinístico o modelo é ao gerar uma resposta. Se você está pedindo uma resposta onde só há uma resposta correta, então você vai querer configurar esses valores mais baixos. Se você está buscando respostas mais diversas, pode querer configurá-los mais altos. O erro número um que as pessoas cometem com essas configurações é supor que são controles de "inteligência" ou "criatividade".


Fonte: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Enviar!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Repita a mesma chamada, como os resultados se comparam?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Resumir Texto  
#### Desafio  
Resuma o texto adicionando um 'tl;dr:' ao final de uma passagem de texto. Observe como o modelo entende como realizar diversas tarefas sem instruções adicionais. Você pode experimentar com prompts mais descritivos que tl;dr para modificar o comportamento do modelo e personalizar o resumo que você recebe(3).  

Trabalhos recentes demonstraram ganhos substanciais em muitas tarefas e benchmarks de PLN ao pré-treinar em um grande corpus de texto seguido de ajuste fino em uma tarefa específica. Embora tipicamente agnóstico à tarefa na arquitetura, esse método ainda requer conjuntos de dados de ajuste fino específicos de tarefas com milhares ou dezenas de milhares de exemplos. Em contraste, humanos geralmente conseguem realizar uma nova tarefa de linguagem a partir de apenas alguns exemplos ou de instruções simples - algo que os sistemas atuais de PLN ainda têm grande dificuldade em fazer. Aqui mostramos que aumentar a escala dos modelos de linguagem melhora muito o desempenho agnóstico à tarefa e com poucos exemplos, às vezes até alcançando competitividade com abordagens anteriores de ajuste fino de última geração. 



Tl;dr


# Exercícios para vários casos de uso  
1. Resumir Texto  
2. Classificar Texto  
3. Gerar Novos Nomes de Produtos


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Classificar Texto  
#### Desafio  
Classifique itens em categorias fornecidas no momento da inferência. No exemplo a seguir, fornecemos tanto as categorias quanto o texto a ser classificado no prompt(*playground_reference). 

Consulta do Cliente: Olá, uma das teclas do teclado do meu laptop quebrou recentemente e eu precisarei de um substituto:

Categoria classificada:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Gerar Novos Nomes de Produto
#### Desafio
Crie nomes de produtos a partir de palavras exemplos. Aqui incluímos no prompt informações sobre o produto para o qual vamos gerar nomes. Também fornecemos um exemplo semelhante para mostrar o padrão que desejamos receber. Também definimos o valor da temperatura alto para aumentar a aleatoriedade e respostas mais inovadoras.

Descrição do produto: Uma máquina de milkshake caseira
Palavras-chave: rápido, saudável, compacto.
Nomes de produtos: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Descrição do produto: Um par de sapatos que serve para qualquer tamanho de pé.
Palavras-chave: adaptável, ajustável, omni-ajuste.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Referências  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Melhores práticas para ajuste fino de modelos GPT para classificar texto](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Para mais ajuda  
[Equipe de Comercialização da OpenAI](AzureOpenAITeam@microsoft.com) 


# Colaboradores
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
